# Module 08 — Proactive Agents

> **SDKs:** `dataclasses`, `hashlib`, `datetime`, `heapq`

| Part | Topic |
|------|-------|
| **1** | Event Deduplication — Redis TTL-based signature cache |
| **2** | Hysteresis & Cooldowns — preventing metric flapping |
| **3** | Quiet Hours & Preferences — user-scoped routing |


---
## Part 1 — Event Deduplication

A naive proactive agent calls the LLM every time a metric crosses a threshold. Without deduplication, the same alert fires 847 times in 30 minutes.

In [1]:
import hashlib, time, json
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class AlertEvent:
    service: str
    metric: str
    value: float
    threshold: float
    severity: str
    timestamp: float = field(default_factory=time.time)
    
    def signature(self) -> str:
        """Stable fingerprint for deduplication. TTL-keyed in Redis."""
        key = f"{self.service}:{self.metric}:{self.severity}:{int(self.threshold)}"
        return hashlib.sha256(key.encode()).hexdigest()[:16]

class AlertDeduplicator:
    """Simulates a Redis TTL-based deduplication cache."""
    def __init__(self, ttl_seconds: int = 300):
        self.ttl = ttl_seconds
        self._seen: dict[str, float] = {}   # sig → expiry_time
        self.stats = {"fired": 0, "deduplicated": 0}

    def should_fire(self, event: AlertEvent) -> bool:
        sig = event.signature()
        now = time.time()
        if sig in self._seen and self._seen[sig] > now:
            self.stats["deduplicated"] += 1
            return False
        self._seen[sig] = now + self.ttl
        self.stats["fired"] += 1
        return True

# ─── Demo ─────────────────────────────────────────────────────────────────────
dedup = AlertDeduplicator(ttl_seconds=300)
EVENTS = [AlertEvent("checkout-ui", "error_rate", v/100, 0.25, "HIGH") for v in range(27, 52)]

print("🔔  Alert Deduplication Demo")
print("=" * 60)
print(f"  Simulating {len(EVENTS)} metric crossings in 30 seconds...")
print(f"  (same service+metric+severity — should only alert ONCE)\n")

for i, evt in enumerate(EVENTS, 1):
    if dedup.should_fire(evt):
        print(f"  ✅  Alert FIRED  [sig={evt.signature()}]  error_rate={evt.value:.2f}")
    else:
        print(f"  ⏸️  Deduplicated ({i})")

print(f"\n  Stats: fired={dedup.stats['fired']}  deduplicated={dedup.stats['deduplicated']}")
print(f"  LLM invocations saved: {dedup.stats['deduplicated']}  (~${dedup.stats['deduplicated'] * 0.012:.2f} saved)")


🔔  Alert Deduplication Demo
  Simulating 25 metric crossings in 30 seconds...
  (same service+metric+severity — should only alert ONCE)

  ✅  Alert FIRED  [sig=51dcfdbe0b6d9e4a]  error_rate=0.27
  ⏸️  Deduplicated (2)
  ⏸️  Deduplicated (3)
  ⏸️  Deduplicated (4)
  ⏸️  Deduplicated (5)
  ⏸️  Deduplicated (6)
  ⏸️  Deduplicated (7)
  ⏸️  Deduplicated (8)
  ⏸️  Deduplicated (9)
  ⏸️  Deduplicated (10)
  ⏸️  Deduplicated (11)
  ⏸️  Deduplicated (12)
  ⏸️  Deduplicated (13)
  ⏸️  Deduplicated (14)
  ⏸️  Deduplicated (15)
  ⏸️  Deduplicated (16)
  ⏸️  Deduplicated (17)
  ⏸️  Deduplicated (18)
  ⏸️  Deduplicated (19)
  ⏸️  Deduplicated (20)
  ⏸️  Deduplicated (21)
  ⏸️  Deduplicated (22)
  ⏸️  Deduplicated (23)
  ⏸️  Deduplicated (24)
  ⏸️  Deduplicated (25)

  Stats: fired=1  deduplicated=24
  LLM invocations saved: 24  (~$0.29 saved)


---
## Part 2 — Hysteresis & Cooldowns: Preventing Flapping

A metric oscillating just above/below a threshold will cause the agent to flip between `FIRING` and `RESOLVED` hundreds of times per hour (flapping). Hysteresis requires the metric to be clearly above/below the threshold for N consecutive readings.

In [2]:
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class HysteresisGate:
    """
    Implements a two-threshold Schmitt trigger for alert flapping prevention.
    - alert_threshold: value must EXCEED this to enter FIRING state
    - resolve_threshold: value must DROP BELOW this to return to NORMAL state
    - consecutive: readings required at each level to confirm state change
    """
    alert_threshold: float
    resolve_threshold: float
    consecutive: int = 3

    _state: Literal["NORMAL", "FIRING"] = field(default="NORMAL", init=False)
    _streak: int = field(default=0, init=False)
    _history: list[str] = field(default_factory=list, init=False)

    def feed(self, value: float) -> str:
        prev_state = self._state
        if self._state == "NORMAL":
            if value > self.alert_threshold:
                self._streak += 1
                if self._streak >= self.consecutive:
                    self._state = "FIRING"
                    self._streak = 0
            else:
                self._streak = 0
        elif self._state == "FIRING":
            if value < self.resolve_threshold:
                self._streak += 1
                if self._streak >= self.consecutive:
                    self._state = "NORMAL"
                    self._streak = 0
            else:
                self._streak = 0

        event = "🔔 FIRED" if self._state == "FIRING" and prev_state == "NORMAL" else (
                "✅ RESOLVED" if self._state == "NORMAL" and prev_state == "FIRING" else
                f"  {self._state:<8}")
        self._history.append(event)
        return self._state

gate = HysteresisGate(alert_threshold=0.30, resolve_threshold=0.20, consecutive=3)

# Simulate noisy metric oscillating around the threshold
values = [0.15, 0.22, 0.28, 0.32, 0.31, 0.33, 0.35, 0.29, 0.27, 0.26,
          0.24, 0.18, 0.17, 0.16, 0.28, 0.31, 0.34, 0.29, 0.22, 0.19]

print("📈  Hysteresis Gate Demo")
print("=" * 60)
print(f"  Alert threshold : > {gate.alert_threshold:.2f} for {gate.consecutive} readings")
print(f"  Resolve threshold: < {gate.resolve_threshold:.2f} for {gate.consecutive} readings")
print()
print(f"  {'t':<5} {'value':<10} {'gate_output'}")
print(f"  {'─'*5} {'─'*10} {'─'*15}")

for i, v in enumerate(values, 1):
    state = gate.feed(v)
    event = gate._history[-1]
    bar = "█" * int(v * 30) + ("!" if state == "FIRING" else "")
    print(f"  {i:<5} {v:.2f}  {bar:<12} {event}")


📈  Hysteresis Gate Demo
  Alert threshold : > 0.30 for 3 readings
  Resolve threshold: < 0.20 for 3 readings

  t     value      gate_output
  ───── ────────── ───────────────
  1     0.15  ████           NORMAL  
  2     0.22  ██████         NORMAL  
  3     0.28  ████████       NORMAL  
  4     0.32  █████████      NORMAL  
  5     0.31  █████████      NORMAL  
  6     0.33  █████████!   🔔 FIRED
  7     0.35  ██████████!    FIRING  
  8     0.29  ████████!      FIRING  
  9     0.27  ████████!      FIRING  
  10    0.26  ███████!       FIRING  
  11    0.24  ███████!       FIRING  
  12    0.18  █████!         FIRING  
  13    0.17  █████!         FIRING  
  14    0.16  ████         ✅ RESOLVED
  15    0.28  ████████       NORMAL  
  16    0.31  █████████      NORMAL  
  17    0.34  ██████████     NORMAL  
  18    0.29  ████████       NORMAL  
  19    0.22  ██████         NORMAL  
  20    0.19  █████          NORMAL  


---
## Part 3 — Quiet Hours & Routing Preferences

Different alerts have different urgency. A 3 AM SEV3 alert should not wake an engineer. It should route to the morning email digest. User-scoped routing ensures agents respect human working hours.

In [3]:
from dataclasses import dataclass
from datetime import datetime, timezone, time as dtime
from typing import Literal, Optional

@dataclass
class UserPreferences:
    user_id: str
    timezone: str          # simplified: "UTC" | "US_EAST" | "EU_CENTRAL"
    quiet_start: dtime     # 22:00 local
    quiet_end: dtime       # 08:00 local
    min_sev_for_page: int  # 1=SEV1 only, 2=SEV1+2, 3=all

@dataclass
class AlertRoute:
    channel: Literal["pagerduty", "slack", "email_digest"]
    reason: str
    deferred_to: Optional[str] = None

def route_alert(
    severity: int,
    alert_title: str,
    user: UserPreferences,
    current_utc_hour: int,   # 0-23
) -> AlertRoute:
    """
    Deterministic routing: never uses an LLM to decide whether to wake a person up.
    This is control-plane logic — it must be provably correct.
    """
    # Convert UTC to user's local hour (simplified)
    tz_offset = {"UTC": 0, "US_EAST": -5, "EU_CENTRAL": 1}
    local_hour = (current_utc_hour + tz_offset.get(user.timezone, 0)) % 24
    
    in_quiet_hours = local_hour >= user.quiet_start.hour or local_hour < user.quiet_end.hour
    
    if severity == 1:
        # SEV1: always wake the person, no exceptions
        return AlertRoute("pagerduty", "SEV1 always pages regardless of quiet hours")
    
    if in_quiet_hours:
        if severity <= user.min_sev_for_page:
            return AlertRoute("email_digest", "In quiet hours — deferred to morning digest",
                              deferred_to=f"08:00 {user.timezone}")
        else:
            return AlertRoute("email_digest", f"SEV{severity} below page threshold",
                              deferred_to=f"08:00 {user.timezone}")
    
    if severity <= user.min_sev_for_page:
        return AlertRoute("slack", f"Business hours — routing to Slack for SEV{severity}")
    
    return AlertRoute("pagerduty", f"SEV{severity} meets page threshold")

# ─── Demo ─────────────────────────────────────────────────────────────────────
alice = UserPreferences("alice", "EU_CENTRAL", dtime(22,0), dtime(8,0), min_sev_for_page=2)

test_cases = [
    (1, "Checkout service DOWN — 100% error rate", 3),    # 3 AM UTC = 4 AM local → quiet
    (2, "EU conversion drop 38%",                  3),    # 3 AM — sev2, quiet hours
    (2, "EU conversion drop 38%",                  14),   # 2 PM — business hours
    (3, "Minor cache miss spike",                  3),    # 3 AM — sev3, quiet hours
    (3, "Minor cache miss spike",                  14),   # 2 PM — business hours
]

print("📬  Quiet Hours & Preference Routing Demo (user: alice, EU_CENTRAL)")
print("=" * 70)
print(f"  {'Severity':<12} {'Time (UTC)':<12} {'Channel':<15} Reason")
print(f"  {'─'*12} {'─'*12} {'─'*15} {'─'*35}")

for sev, title, utc_hour in test_cases:
    route = route_alert(sev, title, alice, utc_hour)
    deferred = f"(deferred to {route.deferred_to})" if route.deferred_to else ""
    print(f"  SEV{sev:<9} {utc_hour:02d}:00 UTC    {route.channel:<15} {route.reason[:35]} {deferred}")


📬  Quiet Hours & Preference Routing Demo (user: alice, EU_CENTRAL)
  Severity     Time (UTC)   Channel         Reason
  ──────────── ──────────── ─────────────── ───────────────────────────────────
  SEV1         03:00 UTC    pagerduty       SEV1 always pages regardless of qui 
  SEV2         03:00 UTC    email_digest    In quiet hours — deferred to mornin (deferred to 08:00 EU_CENTRAL)
  SEV2         14:00 UTC    slack           Business hours — routing to Slack f 
  SEV3         03:00 UTC    email_digest    SEV3 below page threshold (deferred to 08:00 EU_CENTRAL)
  SEV3         14:00 UTC    pagerduty       SEV3 meets page threshold 
